In [ ]:
import pandas as pd
import glob
import numpy as np
import os 
import sys


In [ ]:
masterDir = "/Users/danielruiz/Downloads/Alkenes/methodMap1/deltaDir"

fukuiDir = "/Volumes/KINGSTON/fukui"

fukuiDFs = glob.glob(fukuiDir + "/*.csv")

fukuiDFs = [pd.read_csv(f) for f in fukuiDFs]
combinedFukuiDF = pd.concat(fukuiDFs, axis=0, ignore_index=True, join="inner")
print(combinedFukuiDF[:100])

In [ ]:
print(list(combinedFukuiDF.columns))

In [ ]:
def convertCanonical(str):
    from rdkit import Chem
    mol = Chem.MolFromSmiles(str)
    canonical = Chem.MolToSmiles(mol, isomericSmiles=True, canonical=True)
    return canonical
def fukuiTransfer(inputDF, masterDF, dropCols: list):
    newCols = [col for col in masterDF.columns if col not in dropCols]
    for col in newCols:
        inputDF[col] = "nan"
    for index, row in inputDF.iterrows():
        smiles = row["SMILES"]
        canonical = convertCanonical(smiles)
        if canonical in masterDF["Canonicals"].values:
            idx = masterDF.index[masterDF["Canonicals"] == canonical].tolist()[0]
            matchingRow = masterDF.loc[idx]
            for col in newCols:
                inputDF.at[index, col] = matchingRow[col]
    return inputDF


In [ ]:
dirs = glob.glob(masterDir + "/*.csv")
outputDir = "/Volumes/KINGSTON/fukui/mulliken"
for dir_ in dirs:
    df = pd.read_csv(dir_)
    newDF = fukuiTransfer(df , combinedFukuiDF , ["SMILES" , "Canonicals"])
    saveStr = dir_.split("/")[-1].split(".")[0]
    df.to_csv(outputDir + "/" + saveStr + ".csv", index=False)
